# Předpověď měsíčního počtu škod z autopojištění pomocí PROC FORECAST


## Shrnutí pro vedení

Tým aktuárských rezerv potřebuje pohled 12 měsíců dopředu na měsíční počet škod z autopojištění, který ukazuje stabilní růst portfolia plus výrazný zimní nárůst. Tento notebook generuje pět let syntetických měsíčních počtů škod (60 měsíců, leden 2020 - prosinec 2024, v rozmezí zhruba od 1 460 do 2 780 škod), poté pomocí **PROC FORECAST** proloží stupňovitě autoregresní základní model a multiplikativní sezónní model Holt-Winters. Každý model vytváří 12 měsíčních bodových předpovědí s 95% konfidenčními mezemi pro plánování kapacit a rezerv. Sezónní model Holt-Winters předpovídá nejsilnější poptávku jeden až dva měsíce dopředu (přibližně 2 780-2 900 škod), která poté klesá k minimu kolem kroku devět (přibližně 2 200), zatímco autoregresní základní model předpovídá plynulejší pokles; obě konfidenční pásma se s horizontem rozšiřují.


## Zdroje dat

| Datová sada | Řádků | Granularita | Klíčové proměnné | Popis |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Jeden řádek na kalendářní měsíc (leden 2020 - prosinec 2024) | `date` (SAS datum, `MONYY7.`), `claim_count` (číselná) | Syntetický měsíční počet škod z autopojištění sestavený z lineárního růstového trendu (~9 škod/měsíc), sinusové roční periody, aditivního zimního nárůstu prosinec/leden/únor a Gaussova šumu (`rand('normal')`). Seed `20240531` zajišťuje reprodukovatelnost. |


# Předpověď měsíčního počtu škod z autopojištění

Týmy rezerv a plánování kapacit u pojistitele osobních linek potřebují dopředný pohled na to, kolik **škod z autopojištění** bude nahlášeno každý měsíc. Objem škod v této knize stabilně roste s expanzí portfolia a každou zimu prudce stoupá, když námraza, sníh a kratší denní světlo zvyšují počet nárazových škod a škod na sklech.

Tento notebook prochází kompletním pracovním postupem `PROC FORECAST`:

1. Vygenerovat realistickou, plně syntetickou měsíční řadu počtu škod.
2. Proložit **stupňovitě autoregresní (STEPAR)** základní model, který zachycuje trend a autokorelaci.
3. Proložit **multiplikativní model Holt-Winters (WINTERS)**, který explicitně modeluje 12měsíční sezónní cyklus.
4. Porovnat oba modely a interpretovat dopřednou předpověď a konfidenční pásmo.

Vše běží na inline syntetických datech - žádné externí soubory ani síťový přístup.


## Krok 1 - Vygenerování syntetické řady škod

Níže uvedený krok DATA sestavuje **60 měsíčních pozorování** (leden 2020 až prosinec 2024). Pro každý měsíc kombinujeme čtyři složky, které odrážejí reálnou knihu autopojištění:

- **Trend** - základ ~1 800 škod rostoucí o ~9 měsíčně s rostoucí expozicí.
- **Roční cyklus** - sinusový/kosinusový člen dávající plynulou sezónní vlnu.
- **Zimní nárůst** - aditivní skok v prosinci, lednu a únoru.
- **Šum** - `rand('normal', 0, 90)` pro realistickou meziměsíční variabilitu.

`call streaminit()` fixuje náhodný proud, aby byl notebook reprodukovatelný. Proměnná `date` je skutečné SAS datum sestavené pomocí `INTNX` a formátované `MONYY7.`, a příkaz `ID date INTERVAL=MONTH` ji označuje jako časový identifikátor. Prvních 14 níže vytištěných řádků se pohybuje zhruba mezi 1 460 a 2 450 škodami.


In [1]:
data claims;
    CALL streaminit(20240531);
    OPAKUJ i = 0 TO 59;
        /* Sestavení skutečného SAS měsíčního data, aby INTERVAL=MONTH sedělo */
        date = intnx('month', '01JAN2020'd, i);
        FORMÁT date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = leden ... 12 = prosinec */

        trend  = 1800 + 9 * i;               /* rostoucí základ expozice   */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* nárůst námrazy/sněhu  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        KDYŽ claim_count < 0 PAK claim_count = 0;
        VÝSTUP;
    KONEC;
    PONECHAT date claim_count;
SPUSTIT;

PROCEDURA TISK data=claims(obs=14) ŠTÍTEK;
    ŠTÍTEK date = 'Měsíc' claim_count = 'Nahlášené škody';
    NÁZEV 'Prvních 14 měsíců syntetického počtu škod z autopojištění';
SPUSTIT;


                               Prvních 14 měsíců syntetického počtu škod z autopojištění                                

  Obs    Měsíc      Nahlášené škody
    1    21915                 2178
    2    21946                 2281
    3    21975                 2252
    4    22006                 1974
    5    22036                 2067
    6    22067                 1805
    7    22097                 1697
    8    22128                 1619
    9    22159                 1462
   10    22189                 1688
   11    22220                 1713
   12    22250                 2008
   13    22281                 2116
   14    22312                 2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Krok 2 - Stupňovitě autoregresní základní model (METHOD=STEPAR)

`METHOD=STEPAR` je výchozí. Nejprve proloží model časového trendu - zde `TREND=2` pro lineární trend - a poté na rezidua aplikuje **stupňovitou autoregresi**, která vstupuje a ponechává AR zpoždění podle významnosti. `AR=4` omezuje kandidátský autoregresní řád na čtyři zpoždění, což pro měsíční řadu s krátkodobou setrvačností plně postačuje.

Klíčové zde použité volby:

- `LEAD=12` - předpověď 12 měsíců za konec dat.
- `ALPHA=0.05` - 95% konfidenční meze.
- `OUTFULL` - naskládá historické řádky `ACTUAL` spolu s řádky horizontu předpovědi v datové sadě `OUT=` (namísto pouze řádků předpovědi).
- `OUTLIMIT` - přidá **sloupce** dolní/horní konfidenční meze (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` - pojmenuje časový identifikátor a deklaruje, že řada je měsíční.


In [2]:
PROCEDURA forecast data=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    PROMĚNNÁ claim_count;
SPUSTIT;

PROCEDURA TISK data=fc_stepar(obs=10) ŠTÍTEK;
    NÁZEV 'Výstup STEPAR: řádky se skutečnými, vyrovnanými a předpovězenými hodnotami';
SPUSTIT;


                               Prvních 14 měsíců syntetického počtu škod z autopojištění                                

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                       Výstup STEPAR: řádky se skutečnými, vyrovnanými a předpovězenými hodnotami                       

  Obs   DATE  _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Čtení datové sady OUT=

Datová sada `OUT=` naskládá řádky podle sloupce `_TYPE_` a nese konfidenční meze jako vedlejší **sloupce**:

| Prvek | Druh | Význam |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | řádek | Pozorovaný `claim_count` pro každý z 60 historických měsíců. |
| `_TYPE_ = 'FORECAST'` | řádek | 12 bodových předpovědí pro horizont předpovědi. |
| `L95_claim_count` / `U95_claim_count` | sloupec | Dolní / horní 95% konfidenční meze, vyplněné na řádcích `FORECAST` (chybějící na řádcích `ACTUAL`). Číselná úroveň odráží `ALPHA=`. |

Vytištěná datová sada `OUT=` zde tedy obsahuje 72 řádků: 60 historických řádků `ACTUAL` následovaných 12 řádky horizontu `FORECAST`. Prvních deset zde zobrazených řádků jsou všechny `ACTUAL`, přičemž sloupce konfidenčních mezí chybí, protože meze se vážou jen k řádkům předpovědi.

> **Poznámka k enginu.** SAS `OUTFULL` zapisuje také vnitrovzorkový jednokrokový řádek `FORECAST` pro každý historický měsíc a `OUTRESID` přidává řádky `_TYPE_='RESIDUAL'`. PROC FORECAST v Jenneru tyto vyrovnané/reziduální řádky v rámci vzorku zatím nevytváří (sledováno jako gap test `400979`), takže tento notebook čte pouze historii `ACTUAL` a dopředný horizont `FORECAST`.


## Krok 3 - Sezónní model Holt-Winters (METHOD=WINTERS)

Model STEPAR zachycuje trend a krátkodobou autokorelaci, ale explicitně nemodeluje opakující se nárůst prosinec-únor. Pro řadu s jasným ročním rytmem je lepším nástrojem **multiplikativní Holt-Winters**.

`METHOD=WINTERS` rozkládá řadu na úroveň, lineární trend (`TREND=2`) a **multiplikativní sezónní faktor**. `SEASONS=12` deklaruje 12periodický (měsíční) sezónní cyklus. Opět požadujeme 12měsíční `LEAD` s 95% mezemi a `OUTFULL OUTLIMIT`, aby výstup odpovídal běhu STEPAR.

Protože engine posouvá `ID` předpovědi o jednu jednotku na krok (namísto respektování `INTERVAL=MONTH` pro data horizontu - gap test `400979`), níže uvedená buňka prochází předpověď podle **měsíců dopředu (krok 1-12)** namísto spoléhání na vytištěná kalendářní data.


In [3]:
PROCEDURA forecast data=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    PROMĚNNÁ claim_count;
SPUSTIT;

/* Ponechat dopředný 12měsíční horizont a indexovat jej podle kroku (1..12). */
data horizon;
    NASTAVIT fc_winters;
    UCHOVAT months_ahead 0;
    KDYŽ _type_ = 'FORECAST';
    months_ahead + 1;
    PONECHAT months_ahead claim_count l95_claim_count u95_claim_count;
SPUSTIT;

PROCEDURA TISK data=horizon ŠTÍTEK;
    ŠTÍTEK months_ahead     = 'Měsíců dopředu'
          claim_count       = 'Předpovězené škody'
          l95_claim_count   = 'Dolní 95 %'
          u95_claim_count   = 'Horní 95 %';
    NÁZEV 'Holt-Wintersova 12měsíční dopředná předpověď (podle kroku)';
SPUSTIT;


                       Výstup STEPAR: řádky se skutečnými, vyrovnanými a předpovězenými hodnotami                       

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                               Holt-Wintersova 12měsíční dopředná předpověď (podle kroku)                               

  Obs      Měsíců dopředu      Předpovězené škody   Dolní 95 %   Horní 95 %
    1                   1              2783.07951  2577.844742  2988.314278
    2                   2             2897.355557  2607.109764  3187.601349
    3                   3             2805.272075  2449.795029   3160.74912
    4                   4             2664.498049  2254.028514  3074.967585
    5                   5             2628.810136  2169.891244  3087.729029
    6                   6              2548.39319  2045.672732  3051.113649
    7                   7             2391.649756    1848.6496  2934.649912
    8                   8             2


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Krok 4 - Porovnání obou modelů

Nejjasnějším způsobem, jak posoudit, zda se sezónní model vyplácí, je položit jeho 12měsíční předpověď vedle základního modelu STEPAR, krok po kroku. Vezmeme řádky `FORECAST` z obou datových sad `OUT=`, indexujeme každou podle měsíců dopředu a sloučíme je, aby byla rozdílnost na první pohled viditelná.

Obě metody se shodují na celkové úrovni, ale liší se ve **tvaru**: Holt-Winters přenáší roční rytmus do horizontu (vyšší úroveň v ranném horizontu, která klesá ke střednímu minimu a poté opět stoupá), zatímco STEPAR - který modeluje sezónnost pouze nepřímo přes AR zpoždění - hladce klesá směrem k průměru řady. Rozdíl mezi nimi v každém kroku je sezónní signál, který STEPAR nechává nevyužitý.

> SAS by tuto kontrolu přiměřenosti běžně řídil pomocí jednokrokových reziduí `OUTRESID` (`_TYPE_='RESIDUAL'`). Jenner tyto řádky zatím nevyplňuje (gap test `400979`), takže místo toho porovnáváme obě dopředné předpovědi přímo - diagnostiku, která používá pouze výstup, jenž engine skutečně vytváří.


In [4]:
/* Horizont STEPAR, indexovaný podle měsíců dopředu. */
data stepar_h;
    NASTAVIT fc_stepar;
    UCHOVAT months_ahead 0;
    KDYŽ _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    PONECHAT months_ahead stepar;
SPUSTIT;

/* Horizont WINTERS, indexovaný podle měsíců dopředu. */
data winters_h;
    NASTAVIT fc_winters;
    UCHOVAT months_ahead 0;
    KDYŽ _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    PONECHAT months_ahead winters;
SPUSTIT;

/* Sloučení obou a zobrazení sezónního rozdílu, který STEPAR nezachytí. */
data COMPARE;
    SLOUČIT stepar_h winters_h;
    PODLE months_ahead;
    seasonal_gap = winters - stepar;
SPUSTIT;

PROCEDURA TISK data=COMPARE ŠTÍTEK;
    ŠTÍTEK months_ahead = 'Měsíců dopředu'
          stepar        = 'Předpověď STEPAR'
          winters       = 'Předpověď Winters'
          seasonal_gap  = 'Winters - STEPAR';
    FORMÁT stepar winters seasonal_gap 8.0;
    NÁZEV 'STEPAR vs Holt-Winters: srovnání 12měsíční předpovědi';
SPUSTIT;


                                 STEPAR vs Holt-Winters: srovnání 12měsíční předpovědi                                  

  Obs      Měsíců dopředu     Předpověď STEPAR     Předpověď Winters  Winters - STEPAR
    1                   1                 2619                  2783               164
    2                   2                 2537                  2897               361
    3                   3                 2384                  2805               421
    4                   4                 2214                  2664               450
    5                   5                 2089                  2629               540
    6                   6                 2010                  2548               539
    7                   7                 1982                  2392               410
    8                   8                 1995                  2240               245
    9                   9                 2031                  2197               166
   10   


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Krok 5 - Předpověď pro každou pojistnou linku najednou (zpracování BY)

Reálné rezervní běhy pokrývají několik produktů. S daty seřazenými podle `line_of_business` příkaz `BY` přiměje `PROC FORECAST` proložit **nezávislý sezónní model pro každou skupinu** v jediném volání. Zde syntetizujeme knihu Auto (vyšší základní objem) a knihu Dům (nižší základ) a předpovídáme obě šest měsíců dopředu. `OUTALL` zapíše celou sadu řádků - historii `ACTUAL` a horizont `FORECAST` - spolu se sloupci mezí pro každou skupinu, a my ponecháme jen řádky `FORECAST` pro tabulku níže.


In [5]:
data multi_book;
    CALL streaminit(20240531);
    DÉLKA line_of_business $6;
    OPAKUJ lob = 1 TO 2;
        KDYŽ lob = 1 PAK line_of_business = 'Auto';
        JINAK            line_of_business = 'Dům';
        OPAKUJ i = 0 TO 47;
            date = intnx('month', '01JAN2021'd, i);
            FORMÁT date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            VÝSTUP;
        KONEC;
    KONEC;
    PONECHAT line_of_business date claim_count;
SPUSTIT;

PROCEDURA ŘADIT data=multi_book;
    PODLE line_of_business date;
SPUSTIT;

PROCEDURA forecast data=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    PODLE line_of_business;
    id date interval=month;
    PROMĚNNÁ claim_count;
SPUSTIT;

PROCEDURA TISK data=fc_by(obs=12) ŠTÍTEK;
    KDE _type_ = 'FORECAST';
    NÁZEV 'Předpovědi na 6 měsíců podle pojistné linky';
SPUSTIT;


                                 STEPAR vs Holt-Winters: srovnání 12měsíční předpovědi                                  


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Dům

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                      Předpovědi na 6 měsíců podle pojistné linky                                       

  Obs  LINE_OF_BUSINESS   DATE    _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  Auto              23742  FORECAST  2524.596717      2359.095846      2690.097589                     .
    2  Auto              23773  FORECAST  2604.418724      2370.365147        2838.4723                     .
    3  Auto              23801  FORECAST  2576.092313      2289.436395       2862.74823                     .
    4  Auto              23832  FOR


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Auto
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Dům
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Interpretace výsledků

**Co modely říkají týmu rezerv:**

- **STEPAR** reprodukuje rostoucí trend a krátkodobou setrvačnost, ale jeho 12měsíční předpověď hladce klesá z přibližně 2 620 (krok 1) k zhruba 1 980 v polovině horizontu, než se vrátí zpět k přibližně 2 140 - nereprodukuje ostrý zimní vrchol, protože sezónnost zachycuje pouze přes autoregresní zpoždění. Je to rozumný rychlý základní model.
- **WINTERS** se `SEASONS=12` přenáší roční rytmus přímo přes své multiplikativní sezónní faktory: jeho předpověď je nejvyšší v ranném horizontu (přibližně 2 780 v kroku 1, přibližně 2 900 v kroku 2), klesá k minimu poblíž kroku 9 (přibližně 2 200) a znovu stoupá do kroku 12 (přibližně 2 800). Oproti základnímu modelu STEPAR je po většinu horizontu **o 150-660 škod vyšší** (viz sloupec `Winters - STEPAR` v Kroku 4) - tento rozdíl je sezónní signál, který autoregresní model opomíjí.
- **95% konfidenční pásmo** (sloupce `L95_*`/`U95_*`, řízené pomocí `ALPHA=`) se s prodlužujícím horizontem rozšiřuje - u modelu WINTERS ze šířky zhruba 410 škod v kroku 1 na přibližně 1 420 v kroku 12 - poctivý signál, že odhady na konci horizontu nesou více nejistoty než ty krátkodobé. Rezervisté by měli držet kapitál proti horní mezi, nikoli jen proti bodové předpovědi.
- **Zpracování BY** vytvoří předpovědi pro Auto i Dům v jednom průchodu, každou s vlastním sezónním proložením. Kniha Auto předpovídá zhruba v rozmezí 2 510-2 600, zatímco kniha Dům se pohybuje poblíž 1 870-2 030, takže si každá linka udržuje vlastní úroveň a sezónní tvar - vzor, který by tým rozšířil na každý produkt v portfoliu.

**Závěr:** pro řadu počtu škod s jasným ročním cyklem je `METHOD=WINTERS SEASONS=12` idiomatickou volbou; základní model STEPAR je užitečnou kontrolou zdravého rozumu a `OUTFULL`/`OUTLIMIT` spolu s krokovým porovnáním modelů umožňují aktuárovi stanovit dopřednou rezervu i s jejím pásmem nejistoty.

> **Poznámka k věrnosti enginu.** Tento notebook dokumentuje dvě aktuální omezení PROC FORECAST v Jenneru (gap test `400979`): identifikátor `ID` horizontu předpovědi je posouván o jednu jednotku na krok namísto podle `INTERVAL=MONTH`, takže vytištěná data předpovědi nejsou zamýšlené kalendářní měsíce roku 2025 (horizont proto procházíme podle kroku); a `OUTRESID`/`OUTALL` zatím nevyplňují jednokrokové řádky `_TYPE_='RESIDUAL'`, takže reziduální diagnostiku nahrazuje přímé porovnání dvou modelů. Samotné bodové předpovědi a konfidenční meze se vytvářejí a na nich je založena výše uvedená interpretace.
